In [ ]:
%pip install transformers pymilvus feast torch -q

In [ ]:
from transformers import BertTokenizer, BertModel, RagTokenizer, RagTokenForGeneration
from kfto_feast_rag import FeastRAGRetriever
from feast import FeatureStore, FeatureView, Entity, Field, ValueType, FileSource, RepoConfig
from pymilvus import connections, CollectionSchema, FieldSchema, DataType, Collection
import numpy as np
import os
import pandas as pd
import torch
from torch.utils.data import Dataset, DataLoader


Sample docs

In [ ]:
docs = [
    {"doc_id": 1, "title": "Kubernetes", "text": "Kubernetes is a container orchestration platform."},
    {"doc_id": 2, "title": "Feast", "text": "Feast is a feature store for ML pipelines."}
]
doc_df = pd.DataFrame(docs)
doc_df.to_csv("../docs.csv", index=False)

Tokenize and add to milvus 

In [ ]:
tokenizer = BertTokenizer.from_pretrained("bert-base-uncased")
model = BertModel.from_pretrained("bert-base-uncased")

with torch.no_grad():
    inputs = tokenizer(doc_df["text"].tolist(), return_tensors="pt", padding=True, truncation=True)
    embeddings = model(**inputs).last_hidden_state[:, 0, :].numpy()

connections.connect()
schema = CollectionSchema([
    FieldSchema(name="doc_id", dtype=DataType.INT64, is_primary=True, auto_id=False),
    FieldSchema(name="embedding", dtype=DataType.FLOAT_VECTOR, dim=768),
])
collection = Collection(name="doc_embeddings", schema=schema)
collection.insert([doc_df["doc_id"].tolist(), embeddings])
collection.create_index("embedding", {"metric_type": "IP", "index_type": "IVF_FLAT", "params": {"nlist": 1024}})
collection.load()

Add metadata to Feast

In [ ]:
!feast init feature_repo
os.chdir("feature_repo")

entity = Entity(name="doc_id", join_keys=["doc_id"], value_type=ValueType.INT64)

feature_view = FeatureView(
    name="docs",
    entities=["doc_id"],
    ttl=None,
    schema=[
        Field(name="title", dtype=ValueType.STRING),
        Field(name="text", dtype=ValueType.STRING),
    ],
    online=True,
)

source = FileSource(path="../docs.csv", event_timestamp_column="event_ts")

config = RepoConfig(
    project="rag_project",
    provider="local",
    online_store={"type": "sqlite"},
    offline_store={"type": "file"},
)

store = FeatureStore(config=config)
store.apply([entity, feature_view])
store.materialize_incremental(end_date=pd.Timestamp.now())

Define Feast RagRetriever based on Transformers RagRetriever Class (add reference url)

In [ ]:
retriever = FeastRAGRetriever(
    question_encoder_tokenizer=tokenizer,
    generator_tokenizer=RagTokenizer.from_pretrained("facebook/rag-token-base"),
    feast_repo_path="../feature_repo",
    milvus_collection_name="doc_embeddings"
)

Fine tune RAG Model

In [ ]:
model = RagTokenForGeneration.from_pretrained("facebook/rag-token-base")
model.set_retriever(retriever)

class DummyDataset(Dataset):
    def __getitem__(self, idx):
        return {
            "input_ids": tokenizer("What is Kubernetes?", return_tensors="pt")["input_ids"].squeeze(),
            "attention_mask": tokenizer("What is Kubernetes?", return_tensors="pt")["attention_mask"].squeeze(),
            "labels": tokenizer("Kubernetes is a", return_tensors="pt")["input_ids"].squeeze()
        }
    def __len__(self): return 10

dataloader = DataLoader(DummyDataset(), batch_size=1)
optimizer = torch.optim.AdamW(model.parameters(), lr=1e-5)

model.train()
for epoch in range(1):
    for batch in dataloader:
        batch = {k: v.cuda() if torch.cuda.is_available() else v for k, v in batch.items()}
        output = model(**batch)
        output.loss.backward()
        optimizer.step()
        optimizer.zero_grad()
        print("Loss:", output.loss.item())

Run inference

In [ ]:
model.eval()
input_ids = tokenizer("What is Feast?", return_tensors="pt")["input_ids"]
output_ids = model.generate(input_ids=input_ids, num_beams=2)
print(tokenizer.decode(output_ids[0], skip_special_tokens=True))